# Medical Inventory Forecasting - Error Analysis

This notebook evaluates prediction errors, identifies top forecasting failures, and analyzes error metrics across demand groups using `src.error_analysis`.


## 1. Import Modules and Load Data/Model


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

REPO_ROOT = Path("..").resolve() if Path("..").joinpath("src").exists() else Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data_processing import clean_product_data
from src.feature_engineering import prepare_ml_dataset, split_features_and_target, create_train_test_split
from src.model_persistence import load_model
from src.error_analysis import (
    calculate_prediction_errors,
    get_top_errors,
    analyze_demand_group_performance,
    plot_actual_vs_predicted,
    plot_residual_histogram,
    plot_demand_group_mae,
    plot_top_errors
)

sns.set_theme(style="whitegrid")

# Load dataset and prepare test split
product_data_path = REPO_ROOT / "data" / "Product_Level_Data_Final.csv"
stock = clean_product_data(pd.read_csv(product_data_path))
ml_model = prepare_ml_dataset(stock)
X, y = split_features_and_target(ml_model)
_, X_test, _, y_test = create_train_test_split(X, y, test_size=0.2, random_state=5)

# Load trained model
model = load_model("models/medical_inventory_gb_model.pkl")
y_pred = model.predict(X_test)


## 2. Prediction Error Calculation


In [ ]:
error_df = calculate_prediction_errors(X_test, y_test, y_pred)
top20 = get_top_errors(error_df, top_n=20)
print("Top 5 Error Instances:")
display(top20[["Actual", "Predicted", "Error", "Absolute_Error"]].head())


## 3. Visualization of Errors


In [ ]:
fig_avp = plot_actual_vs_predicted(y_test, y_pred)
plt.show()

fig_hist = plot_residual_histogram(error_df["Error"])
plt.show()

fig_top20 = plot_top_errors(top20)
plt.show()


## 4. Demand Group Error Breakdown


In [ ]:
demand_perf = analyze_demand_group_performance(y_test, y_pred)
display(demand_perf)

fig_demand = plot_demand_group_mae(demand_perf)
plt.show()
